In [1]:
import os
import torch
import numpy as np
from PIL import Image, ImageEnhance, ImageOps
from realesrgan.utils import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
import pandas as pd
from bs4 import BeautifulSoup, Tag
from collections import defaultdict
import easyocr
# from paddleocr import PaddleOCR
import re
from datetime import datetime
import time
import sys
# import matplotlib.pyplot as plt

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load CLIP model and processor
import clip
clip_model, clip_preprocess = clip.load("ViT-B/32", download_root="Path_to/CLIP-main/clip/model")
clip_model = clip_model.to(device)
print("CLIP model loaded successfully.")

CLIP model loaded successfully.


In [4]:
def read_config_paths(path_file="config_path.txt"):
    path_config = {}
    with open(path_file, "r") as f:
        for line in f:
            if "=" in line:
                key, value = line.strip().split("=", 1)
                path_config[key.strip()] = value.strip()

    variant = path_config["variant"]
    report_path = path_config["report_path_template"].format(variant=variant)
    image_dir = path_config["image_dir_template"].format(variant=variant)
    output_dir = path_config["output_dir"]
    relative_image_path = path_config["relative_image_path_template"].format(variant=variant)

    # Ensure output folder exists
    os.makedirs(output_dir, exist_ok=True)
    
    return {
        "variant": variant,
        "report_path": report_path,
        "image_dir": image_dir,
        "output_dir": output_dir,
        "relative_image_path_for_html": relative_image_path
    }

In [5]:
def read_variant_thresholds(config_file="thresholds.txt", variant_name=""):
    thresholds = {}
    current_variant = None

    with open(config_file, "r") as f:
        for line in f:
            line = line.strip()
            
            # Stop reading once the Threshold meanings section starts
            if line.startswith("# Threshold meanings:"):
                break
            
            # Identify variant block
            if line.startswith("# Variant:"):
                current_variant = line.replace("# Variant:", "").strip()
            elif "=" in line and current_variant == variant_name:
                key, value = line.split("=", 1)
                thresholds[key.strip()] = float(value.strip())

    return thresholds

In [6]:
# Load config paths and thresholds
paths = read_config_paths("config_path.txt")
thresholds = read_variant_thresholds("thresholds.txt", variant_name=paths["variant"])

In [7]:
# === Format variant name ===
def find_variant_display_name():    
    variant_display_name = paths["variant"].replace("_Regression_Tests_globalTestReport", "")
    variant_display_name = variant_display_name.replace("_", " ")

    if "Legacy" in paths["variant"]:
        variant_display_name += " Legacy"

    return variant_display_name

In [8]:
def get_unique_path(base_path):
    """Return a unique path by appending (1), (2), etc. if needed."""
    if not os.path.exists(base_path):
        return base_path

    base, ext = os.path.splitext(base_path)
    counter = 1
    while True:
        new_path = f"{base} ({counter}){ext}"
        if not os.path.exists(new_path):
            return new_path
        counter += 1

In [9]:
# Define paths
image_dir = paths["image_dir"]
relative_image_path = paths["relative_image_path_for_html"]

variant_display_name = find_variant_display_name()
output_dir = paths["output_dir"]

output_csv = get_unique_path(os.path.join(output_dir, f"output_{variant_display_name}.csv"))
output_html = get_unique_path(os.path.join(output_dir, f"aiAnalysisReport_{variant_display_name}.html"))

In [10]:
# Load HTML
with open(paths["report_path"], "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

In [11]:
# Output log
results = []

In [12]:
def compute_scale_mismatch(img1, img2):
    w1, h1 = img1.size
    w2, h2 = img2.size
    w_diff = abs(w1 - w2) / max(w1, w2)
    h_diff = abs(h1 - h2) / max(h1, h2)
    return round((w_diff + h_diff) / 2 * 100, 2)

In [13]:
def compute_clip_mismatch(image1, image2):
    # Skip comparison if image sizes are different
    if image1.size != image2.size:
        return "NA"
    
    # Preprocess images
    image1 = clip_preprocess(image1).unsqueeze(0).to(device)
    image2 = clip_preprocess(image2).unsqueeze(0).to(device)
    
    with torch.no_grad():
        features1 = clip_model.encode_image(image1)
        features2 = clip_model.encode_image(image2)

    # Normalize the features
    features1 /= features1.norm(dim=-1, keepdim=True)
    features2 /= features2.norm(dim=-1, keepdim=True)

    similarity = torch.cosine_similarity(features1, features2).item()
    mismatch_percent = (1 - similarity) * 50
    return round(mismatch_percent, 2)

In [14]:
def compute_clip_mismatch_segmentwise(image1, image2, base_sizes=[224, 128, 64, 32, 16]):
    image1 = image1.convert("RGB")
    image2 = image2.convert("RGB")

    if image1.size != image2.size:
        return "NA", "NA"

    w, h = image1.size
    mismatch_scores = []

    # Dynamically select segment size
    for size in base_sizes:
        if (w // size >= 2 or h // size >= 2) or size == base_sizes[-1]:
            seg_w, seg_h = size, size
            break

    for y in range(0, h, seg_h):
        for x in range(0, w, seg_w):
            seg1 = image1.crop((x, y, min(x + seg_w, w), min(y + seg_h, h)))
            seg2 = image2.crop((x, y, min(x + seg_w, w), min(y + seg_h, h)))

            # Pad to segment size if smaller
            if seg1.size != (seg_w, seg_h):
                seg1 = ImageOps.pad(seg1, (seg_w, seg_h), color=(0, 0, 0))
                seg2 = ImageOps.pad(seg2, (seg_w, seg_h), color=(0, 0, 0))

            # Preprocess for CLIP
            seg1_tensor = clip_preprocess(seg1).unsqueeze(0).to(device)
            seg2_tensor = clip_preprocess(seg2).unsqueeze(0).to(device)

            with torch.no_grad():
                f1 = clip_model.encode_image(seg1_tensor)
                f2 = clip_model.encode_image(seg2_tensor)

            f1 /= f1.norm(dim=-1, keepdim=True)
            f2 /= f2.norm(dim=-1, keepdim=True)

            similarity = torch.cosine_similarity(f1, f2).item()
            mismatch = (1 - similarity) * 50
            mismatch_scores.append(round(mismatch, 2))

    max_mismatch = round(max(mismatch_scores), 2)
    return max_mismatch, mismatch_scores

In [15]:
def enlarger_enhancer(image1, image2, upscale_factor=2, sharpness_factor=5):
    if not isinstance(image1, Image.Image) or not isinstance(image2, Image.Image):
        raise TypeError("Inputs must be PIL.Image objects.")
    
    # Resize images
    new_size = (image1.width * upscale_factor, image1.height * upscale_factor)
    enlarged1 = image1.resize(new_size, Image.BICUBIC)
    enlarged2 = image2.resize(new_size, Image.BICUBIC)

    # Enhance sharpness
    # enhanced1 = ImageEnhance.Sharpness(enlarged1).enhance(sharpness_factor)
    # enhanced2 = ImageEnhance.Sharpness(enlarged2).enhance(sharpness_factor)

    return enlarged1, enlarged2

In [ ]:
# Initialize EasyOCR reader with English and French
reader = easyocr.Reader(
    ['en', 'fr'],
    download_enabled=False,
    model_storage_directory="Path_to/EasyOCR-master/easyocr/weights"
)

# Initialize RRDBNet + RealESRGAN once
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='Path_to/Real-ESRGAN-master/realesrgan/weights/RealESRGAN_x4plus.pth',
    model=model.to(device),
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=False,
)

def realesrgan_enlarger_enhancer(image):
    img_np = np.array(image)
    sr_image_np, _ = upsampler.enhance(img_np)
    return Image.fromarray(np.clip(sr_image_np, 0, 255).astype("uint8"))

def run_Eocr_on_realesrgan_enhncd_image(img_ref, img_check):
    ref = realesrgan_enlarger_enhancer(img_ref).convert("RGB")
    check = realesrgan_enlarger_enhancer(img_check).convert("RGB")

    ref_texts = reader.readtext(np.array(ref), detail=0)
    check_texts = reader.readtext(np.array(check), detail=0)

    ref_text = ' '.join(ref_texts).strip().lower()
    check_text = ' '.join(check_texts).strip().lower()

    if not ref_text and not check_text:
        same_text = 'NA'
    else:
        same_text = ref_text == check_text

    return ref_text, check_text, same_text, ref, check

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [17]:
def determine_status(same_text, clip_mm, sgmnt_clip_max_mm, sgmnt_clip_mm_scores):
    if same_text == 'NA':
       status, color = "Investigate", "orange"
    
    clip_threshold_investigate = thresholds["clip_threshold_investigate"]
    clip_threshold_nok = thresholds["clip_threshold_nok"]
    segment_threshold_investigate = thresholds["segment_threshold_investigate"]
    segment_threshold_nok = thresholds["segment_threshold_nok"]

    if isinstance(sgmnt_clip_max_mm, str) and sgmnt_clip_max_mm == "NA":
        return "Faulty", "gray"

    high_segments = [v for v in sgmnt_clip_mm_scores if v > 1.0]
    single_striking_diff = clip_mm < 1 and len(high_segments) == 1 and sgmnt_clip_max_mm > segment_threshold_nok

    if clip_mm > clip_threshold_nok:
        status, color = "NOK", "red"
    elif clip_mm > clip_threshold_investigate:
        if single_striking_diff:
            return "NOK", "red"
        status, color = "Investigate", "orange"
    else:
        if single_striking_diff:
            status, color = "NOK", "red"
        elif sgmnt_clip_max_mm > segment_threshold_nok:
            status, color = "Investigate", "orange"
        elif sgmnt_clip_max_mm > segment_threshold_investigate:
            status, color = "Investigate", "orange"
        else:
            status, color = "OK", "green"
        
    return status, color

In [18]:
scenario_dict = defaultdict(list)

with open(paths["report_path"], "r", encoding="utf-8") as f:
    lines = f.readlines()

current_scenario = None

for line in lines:
    line = line.strip()
    # Robust: check for "scenario script:" anywhere in the line
    if "<b>scenario script:" in line.lower():
        # Use regex to extract the text after :</b>
        # E.g. <b>scenario script:</b> PATH<br>
        match = re.search(r'scenario script:</b>(.*?)<br>', line, re.IGNORECASE)
        if match:
            current_scenario = match.group(1).strip()
    elif "toCheck" in line and line.lower().endswith((".png<br>", ".jpg<br>")):
        if current_scenario:
            clean_img = line.replace("<br>", "").strip()
            scenario_dict[current_scenario].append(clean_img)

# print(dict(scenario_dict))


**Main Loop** 👇:
1. Runs the models
2. Collects relevant sections/blocks from the html file

In [19]:
pair_count = 0
MAX_PAIRS = 100

# Extract mismatching image pairs
sections = soup.find_all("font", string=lambda s: s in [
    "Result and reference : simple difference",
    "Result and reference : not same size"
])

status_sections = defaultdict(list)
total_sections = len(sections)

os.makedirs("./output", exist_ok=True)
variant_log_path = get_unique_path(f"./output/OCR_log_{variant_display_name}.txt")

print('Processing now...')
start_time = time.time()
print("⚠️Please do not access the output folder until processing is complete.")

for section in sections:
    pair_count += 1
    progress = (pair_count / total_sections) * 100
    elapsed_time = time.time() - start_time
    bar_length = 30  # Length of the visual progress bar
    filled_len = int(bar_length * progress // 100)
    bar = '=' * filled_len + '-' * (bar_length - filled_len)
    print(
    f"\rProcessing [{bar}] {progress:5.1f}% | Elapsed: "
    f"{str(int(elapsed_time)) + 's' if (elapsed_time <= 60) else f'{(elapsed_time/60):.2f}m'}",
    end='',
    flush=True)
    # if pair_count >= MAX_PAIRS:
    #     break

    section['color'] = "darkblue"
    
    images_block = section.find_all_next("img", limit=2)
    if len(images_block) < 2:
        continue

    check_img_src = images_block[0]["src"]
    ref_img_src = images_block[1]["src"]

    check_img_path = os.path.join(image_dir, os.path.basename(check_img_src))
    ref_img_path = os.path.join(image_dir, os.path.basename(ref_img_src))

    #Handling the missing images
    check_img_exists = os.path.exists(check_img_path)
    ref_img_exists = os.path.exists(ref_img_path)
    if not check_img_exists or not ref_img_exists:
        # print(f"Missing image file: {check_img_path if not check_img_exists else ''} {ref_img_path if not ref_img_exists else ''}")
        status = "Faulty"
        color = "gray"

        status_tag = soup.new_tag("p")
        status_tag.string = f"Model Status: {status}"
        status_tag["style"] = f"color:{color}; font-weight:bold"
        images_block[1].insert_after(status_tag)

        results.append({
            "Ref Image": os.path.basename(ref_img_src),
            "Check Image": os.path.basename(check_img_src),
            "Image Size": 'NA',
            "Size Group": 'NA',
            # "Pixel Mismatch %": "NA",
            "CLIP Mismatch %": "NA",
            "Max Segment Mismatch %": "NA",
            "All Segment Mismatch %": "NA",
            "Scale Mismatch %": "NA",
            "Status": status
        })

        # Collect the relevant block (font + 2 imgs + status tag)
        section_block = [section] + [str(images_block[0]["src"])] + [str(images_block[1]["src"])] + images_block + [status_tag]
        status_sections[status].append(section_block)
        continue

    try:
        img_check = Image.open(check_img_path).convert("RGB")
        img_ref = Image.open(ref_img_path).convert("RGB")
        scale_mm = compute_scale_mismatch(img_ref, img_check)

        if scale_mm > 0:
            # Skip other metrics if scale mismatch
            status, color = "Faulty", "gray"
            clip_mm = sgmnt_pixel_max_mm = sgmnt_pixel_mm_scores = pixel_mm = "NA"
        else:
            # Enlarge if needed
            # def plot_images(ref, check, title_suffix):
            #     plt.figure(figsize=(10, 5))
            #     plt.subplot(1, 2, 1)
            #     plt.imshow(ref)
            #     plt.title(f"Enlarged Ref Image {title_suffix}")
            #     plt.axis('off')
            #     plt.subplot(1, 2, 2)
            #     plt.imshow(check)
            #     plt.title(f"Enlarged ToCheck Image {title_suffix}")
            #     plt.axis('off')
            #     plt.tight_layout()
            #     plt.show()

            same_text = True
            was_enlarged = False
            if 'BUZ' in ref_img_src:
                ref_text, check_text, same_text, img_ref, img_check = run_Eocr_on_realesrgan_enhncd_image(img_ref, img_check)
                was_enlarged = True
                log_line = (
                    f"toCheck img name: {os.path.basename(check_img_src)}, "
                    f"ref img text: {ref_text}, "
                    f"toCheck img text: {check_text}, "
                    f"is same text: {same_text}\n"
                )

                with open(variant_log_path, "a", encoding="utf-8") as log_file:
                    log_file.write(log_line)

            if not was_enlarged:
                if img_ref.width < 224 and img_ref.height < 224:
                    img_ref, img_check = enlarger_enhancer(img_ref, img_check)
                    # plot_images(img_ref, img_check, "(x5)")
                # elif img_ref.width <= 112 and img_ref.height <= 112:
                #     img_ref, img_check = enlarger_enhancer(*enlarger_enhancer(img_ref, img_check))
                    # plot_images(img_ref, img_check, "(x25)")                

            clip_mm = compute_clip_mismatch(img_ref, img_check)
            sgmnt_clip_max_mm, sgmnt_clip_mm_scores = compute_clip_mismatch_segmentwise(img_ref, img_check)
            status, color = determine_status(same_text, clip_mm, sgmnt_clip_max_mm, sgmnt_clip_mm_scores) if (same_text) else ('NOK', 'red')
            size_group = "Small" if (img_check.height*img_check.width < (64*64)) else "Big"

        results.append({
            "Ref Image": os.path.basename(ref_img_src),
            "Check Image": os.path.basename(check_img_src),
            "Image Size": img_check.width*img_check.height if scale_mm == 0 else "NA",
            "Size Group": size_group if scale_mm == 0 else "NA",
            "CLIP Mismatch %": clip_mm,
            "Max Segment Mismatch %": sgmnt_clip_max_mm,
            "All Segment Mismatch %": sgmnt_clip_mm_scores,
            "Scale Mismatch %": scale_mm,
            "Status": status
        })

        status_tag = soup.new_tag("p")
        status_tag.string = f"Model Status: {status}"
        status_tag["style"] = f"color:{color}; font-weight:bold"
        images_block[1].insert_after(status_tag)

        # pair_count += 1
        # Store the scenario script info too
        section_block = [section] + [str(images_block[0]["src"])] + [str(images_block[1]["src"])] + images_block + [status_tag]
        status_sections[status].append(section_block)

    except Exception as e:
        print(f"Error comparing {check_img_src} and {ref_img_src}: {e}")

print('\n✂ Formatting the final report, please wait...')

Processing now...
⚠️Please do not access the output folder until processing is complete.
Processing [==============================] 100.0% | Elapsed: 117.74m
✂ Formatting the final report, please wait...


In [20]:
sections_missing_imgs = soup.find_all("font", string=lambda s: s == "Result and reference : missing image")

for section in sections_missing_imgs:
    scenario_tag = section.find_previous(string=re.compile(r"scenario script:", re.IGNORECASE))
    scenario_script = scenario_tag.parent if scenario_tag else None

    section['color'] = "darkblue"
    status = "Faulty"
    color = "gray"
    
    images_block = section.find_all_next("img", limit=2)

    status_tag = soup.new_tag("p")
    status_tag.string = f"Model Status: {status}"
    status_tag["style"] = f"color:{color}; font-weight:bold"
    images_block[1].insert_after(status_tag)

    section_block = [section] + [str(images_block[0]["src"])] + [str(images_block[1]["src"])] + images_block + [status_tag]
    status_sections["Faulty"].append(section_block)

    results.append({
        "Ref Image": os.path.basename(ref_img_src),
        "Check Image": os.path.basename(check_img_src),
        "Image Size": 'NA',
        "Size Group": 'NA',
        # "Pixel Mismatch %": "NA",
        "CLIP Mismatch %": "NA",
        "Max Segment Mismatch %": "NA",
        "All Segment Mismatch %": "NA",
        "Scale Mismatch %": "NA",
        "Status": status
    })

In [21]:
# Update image src in soup
for img_tag in soup.find_all("img"):
    filename = os.path.basename(img_tag["src"])
    new_src = os.path.join(relative_image_path, filename).replace("\\", "/")
    img_tag["src"] = new_src

In [22]:
def decide_color(status):
    if status == 'OK':
        return 'green'
    elif status == 'NOK':
        return 'red'
    elif status == 'Faulty':
        return 'grey'
    elif status == 'Investigate':
        return 'orange'
    return 'black'

Summary Table Creator 👇

In [23]:
# 1. Count status occurrences
status_counts = {
    "OK": 0,
    "NOK": 0,
    "Investigate": 0,
    "Faulty": 0
}
for result in results:
    status = result["Status"]
    if status in status_counts:
        status_counts[status] += 1

# Remove existing duplicate summary tables if they exist
existing_tables = soup.find_all("table", attrs={"border": "1"})
for table in existing_tables:
    if "Status" in table.text and "Count" in table.text:
        table.decompose()

body = soup.body or soup

# === Create header elements ===
soup_html = BeautifulSoup(features="html.parser")

main_heading = soup_html.new_tag("h1", style="text-align:center; color:#003366; margin-bottom:5px;")
main_heading.string = variant_display_name

sub_heading = soup_html.new_tag("h2", style="text-align:center; color:#444444; margin-top:0; margin-bottom:20px;")
sub_heading.string = "AI Global Report Analysis"

summary_title = soup_html.new_tag("h3", style="text-align:center; color:#222222;")
summary_title.string = "Summary Table"

# === Create summary table HTML ===
summary_table = soup_html.new_tag("table", border="1", style="margin-left:auto; margin-right:auto; margin-bottom:20px; border-collapse:collapse;")
header = soup_html.new_tag("tr")

th_status = soup_html.new_tag("th")
th_status.string = "Status"
header.append(th_status)

th_count = soup_html.new_tag("th")
th_count.string = "Count"
header.append(th_count)

summary_table.append(header)

for status, count in status_counts.items():
    row = soup_html.new_tag("tr")
    color = {
        "OK": "green",
        "NOK": "red",
        "Investigate": "orange",
        "Faulty": "gray"
    }[status]

    cell_status = soup_html.new_tag("td", style=f"color:{color}; font-weight:bold; padding:4px 10px;")
    cell_status.string = status

    cell_count = soup_html.new_tag("td", style="padding:4px 10px;")
    cell_count.string = str(count)

    row.append(cell_status)
    row.append(cell_count)
    summary_table.append(row)

# === Extract global generation date ===
def extract_global_generation_date(soup):
    b_tag = soup.find("b", string=re.compile(r"^generation date:$", re.IGNORECASE))
    if b_tag and b_tag.next_sibling:
        date_text = b_tag.next_sibling.strip()
        if re.match(r"\d{2}-[A-Za-z]{3}-\d{4}", date_text):
            return date_text
    return "NA"

global_date = extract_global_generation_date(soup)

# Today's date
today_str = datetime.today().strftime("%d-%b-%Y")

# Date block
date_info_div = soup_html.new_tag("div")

if global_date != "NA":
    global_date_p = soup_html.new_tag("p")
    global_date_p.string = f"Global test report generation date: {global_date}"
    date_info_div.append(global_date_p)

ai_date_p = soup_html.new_tag("p")
ai_date_p.string = f"AI analysis report generation date: {today_str}"
date_info_div.append(ai_date_p)

# === Fix status explanation (remove duplicates) ===
explanation_div = soup_html.new_tag("div", style="margin-bottom:20px;")
explanation_title = soup_html.new_tag("h3")
explanation_title.string = "Status Legend"
explanation_div.append(explanation_title)

status_explanations = {
    "OK": "The images are visually and semantically aligned.",
    "Investigate": "Some mismatches found; human review advised.",
    "NOK": "Clear and significant mismatch between images.",
    "Faulty": "Image comparison skipped due to missing or size mismatch."
}

explanation_list = soup_html.new_tag("ul")
for status, explanation in status_explanations.items():
    li = soup_html.new_tag("li")
    li.string = f"{status}: {explanation}"
    explanation_list.append(li)

explanation_div.append(explanation_list)


# Report marker
report_start = soup_html.new_tag("h3", style="text-align:left; margin-top:30px;")
report_start.string = "Report:"

# === Insert all top content at once ===
insert_items = [
    main_heading,
    sub_heading,
    summary_title,
    summary_table,
    explanation_div,
    report_start,
    date_info_div,
]

for item in reversed(insert_items):  # Insert in reverse to maintain top-to-bottom order
    if isinstance(body.contents[0], Tag):
        body.insert(0, item)
    else:
        body.insert(0, item)

In [24]:
# Save results
pd.DataFrame(results).to_csv(output_csv, index=False)
with open(output_html, "w", encoding="utf-8") as f:
    f.write(str(soup))

Duplicate Status Remover 👇

In [25]:
# Load the annotated HTML
with open(output_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# === Step 1: Remove duplicate model status lines ===
status_tags = soup.find_all("p", string=lambda s: s and s.strip().startswith("Model Status:"))
deductions = {"OK": 0, "NOK": 0, "Investigate": 0, "Faulty": 0}

i = 0
while i < len(status_tags) - 1:
    current = status_tags[i]
    next_tag = status_tags[i + 1]
    if (current.find_next_sibling() == next_tag and
        current.string.strip() == next_tag.string.strip() and
        current.string.strip().startswith("Model Status:")):

        status_text = current.string.strip().split(":")[-1].strip()
        if status_text in deductions:
            deductions[status_text] += 1
        next_tag.decompose()
        status_tags.pop(i + 1)
    else:
        i += 1

# === Step 2: Remove duplicate headers ===
def remove_duplicate_tag(tag_name, match_text):
    """Remove all but the first tag that matches given text."""
    tags = soup.find_all(tag_name, string=lambda s: s and s.strip() == match_text.strip())
    for dup_tag in tags[1:]:
        dup_tag.decompose()

remove_duplicate_tag("h1", soup.find("h1").text if soup.find("h1") else "")
remove_duplicate_tag("h2", "AI Global Report Analysis")
remove_duplicate_tag("h3", "Summary Table")
remove_duplicate_tag("h3", "Status Legend")
remove_duplicate_tag("h3", "Report:")

# Remove duplicate summary tables (keep only first)
summary_tables = soup.find_all("table", attrs={"border": "1"})
if len(summary_tables) > 1:
    for table in summary_tables[1:]:
        table.decompose()

# Remove duplicate <ul> lists under status legend
status_lists = soup.find_all("ul")
if len(status_lists) > 1:
    for ul in status_lists[1:]:
        ul.decompose()

# === Step 3: Adjust counts in the summary table ===
summary_table = soup.find("table", attrs={"border": "1"})
if summary_table:
    rows = summary_table.find_all("tr")[1:]  # Skip header
    for row in rows:
        status_cell, count_cell = row.find_all("td")
        status = status_cell.text.strip()
        if status in deductions and deductions[status] > 0:
            original_count = int(count_cell.text.strip())
            new_count = max(0, original_count - deductions[status])
            count_cell.string = str(new_count)

# Save updated HTML
with open(output_html, "w", encoding="utf-8") as f:
    f.write(str(soup))

# print("✅ Duplicate Model Status entries and headers removed; summary adjusted.")

Lil more formatting and clickable links addition 👇

In [26]:
# Load the annotated HTML
with open(output_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# --- Step 1: Delete everything from body except the summary table ---
body = soup.body or soup
summary_table = body.find("table", attrs={"border": "1"})

# Preserve: h1, h2, h3 (Summary Table), table, div (explanation), h3 (Report:)
def should_preserve(tag):
    if not isinstance(tag, Tag):
        return False
    if tag.name == "h1":
        return True
    if tag.name == "h2":
        return True
    if tag.name == "h3" and tag.text.strip().lower() == "summary table":
        return True
    if tag.name == "table" and tag.has_attr("border") and tag["border"] == "1":
        return True
    if tag.name == "div" and "OK" in tag.text and "Investigate" in tag.text:
        return True
    if tag.name == "h3" and tag.text.strip().lower() == "report:":
        return True
    if tag.name == "div" and "generation date" in tag.text:
        return True
    return False

# Remove everything else
for tag in list(body.contents):
    if not should_preserve(tag):
        tag.extract()

# --- Step 2: Make summary table status names clickable using anchor links ---
for row in summary_table.find_all("tr")[1:]:  # skip header
    status_cell = row.find_all("td")[0]
    status_text = status_cell.text.strip()
    color_text = decide_color(status_text)
    anchor = soup.new_tag("a", href=f"#{status_text}", style=f"color:{color_text};")
    anchor.string = status_text
    status_cell.clear()
    status_cell.append(anchor)

# --- Step 3: Insert grouped blocks section-wise ---
for status, blocks in status_sections.items():
    # Anchor for scroll target
    anchor_tag = soup.new_tag("a", attrs={"name": status})
    body.append(anchor_tag)

    # Section heading    
    color = decide_color(status)

    header_tag = soup.new_tag("h2", style=f"color:{color}; margin-top:30px;")
    header_tag.string = f"{status} Image Comparisons"
    body.append(header_tag)

    # Add all blocks under this status
    for block in blocks:
        for element in block:
            if isinstance(element, Tag):
                body.append(element)
            else:
                # Assume it's an image src string (like "check.png"), wrap in <p>
                img_note = soup.new_tag("p", style="color:gray; font-style:italic")
                img_note.string = f"(Note: Image src reference - {element})"
                body.append(img_note)


# Save updated HTML
with open(output_html, "w", encoding="utf-8") as f:
    f.write(str(soup))

In [27]:
# Load output HTML
with open(output_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# Find all toCheck <img> tags
for img_tag in soup.find_all("img"):
    src = img_tag.get("src", "")
    if "toCheck" not in src:
        continue

    # Find which scenario this image belongs to
    matched_scenario = None
    for scenario, images in scenario_dict.items():
        for img in images:
            img_clean = img.strip().split("\\")[-1]  # images\toCheck_XXX.png → toCheck_XXX.png
            src_clean = src.replace("../input/", "").split("/")[-1]  # handle relative path
            if img_clean in src_clean:
                matched_scenario = scenario
                break
        if matched_scenario:
            break

    if matched_scenario:
        # Insert scenario <p> above this <img>
        scenario_p = soup.new_tag("p")

        # Create the <b> tag for the prefix
        b_tag = soup.new_tag("b")
        b_tag.string = "Scenario Script: "
        b_tag['style'] = "color:#003366;"  # style the bold prefix

        # Append <b> and the rest of the text
        scenario_p.append(b_tag)
        scenario_p.append(matched_scenario)

        # Insert it
        img_tag.insert_before(scenario_p)

# Save updated output
with open(output_html, "w", encoding="utf-8") as f:
    f.write(str(soup))

# print("✅ Scenario tags inserted above matching toCheck images.")


In [28]:
# ----------- STEP 1: Extract KO Segments from globalTestReport -----------

ko_scenarios = defaultdict(list)

with open(paths["report_path"], "r", encoding="utf-8") as f:
    lines = f.readlines()

current_script = None
collect_ko_lines = []
inside_scenario = False

for line in lines:
    line = line.strip()

    # New scenario starts
    if line.lower().startswith("<b>scenario script:"):
        if current_script and collect_ko_lines:
            ko_scenarios[current_script].extend(collect_ko_lines)
        current_script = line.split(":", 1)[1].strip().replace("</b>", "").strip("<br>")
        collect_ko_lines = []
        inside_scenario = True

    elif "effective value" in line:
        collect_ko_lines.append(line)

# Catch last scenario
if current_script and collect_ko_lines:
    ko_scenarios[current_script].extend(collect_ko_lines)


# ----------- STEP 2: Append KO Segments to existing output_html -----------

with open(output_html, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# Add KO section at the bottom
ko_section = soup.new_tag("div")
ko_section.append(soup.new_tag("h2"))
ko_section.h2.string = "Scenario Segments with KO Checks"

for script, ko_lines in ko_scenarios.items():
    # Add each KO line first
    for ko_line in ko_lines:
        soup_line = BeautifulSoup(ko_line, "html.parser")
        clean_text = soup_line.get_text().strip()

        p_ko = soup.new_tag("p")
        p_ko.string = clean_text
        p_ko["style"] = "color:red; font-family:monospace;"
        ko_section.append(p_ko)

    # Then the Scenario Script after KO lines
    p_script = soup.new_tag("p")
    b_tag = soup.new_tag("b")
    b_tag.string = "Scenario Script: "
    p_script.append(b_tag)
    p_script.append(script)
    ko_section.append(p_script)

# Safe append
if soup.body:
    soup.body.append(ko_section)
else:
    body_tag = soup.new_tag("body")
    body_tag.append(ko_section)
    soup.append(body_tag)

# Write back updated HTML
with open(output_html, "w", encoding="utf-8") as f:
    f.write(str(soup))

print("\n✅ Processing complete. You may now access the output folder.")
# input("Press Enter to exit...")
# sys.exit(1)



✅ Processing complete. You may now access the output folder.
